# Laya parity runs (Kaggle, 2 x T4)

Runs one stage of the Laya side of the Kev-Laya parity phase
(`docs/specs/kev_laya_parity_protocol.md`, `configs/laya_parity.yaml`). Stage `sweep`: 7 domain
runs. Stage `finals`: seeds of the two selected settings, the joint cell and the held-out-table
run (11 runs). Each GPU runs its queue one run after another; every run is saved, predicted and
marked `DONE` the moment it ends, so a rerun of the notebook skips finished runs.

Settings: Accelerator **GPU T4 x2**, Internet **on**, Add Input > the private dataset
`group4-laya-parity` (the version that holds `laya_parity_plan_<STAGE>.json`). For an unattended
run use **Save Version > Save & Run All**.

Outputs: `/kaggle/working/predictions/<run>/` (small; download first),
`/kaggle/working/laya_parity/<run>/` (checkpoints, about 0.85 GB each),
`/kaggle/working/predictions_<STAGE>.zip`. Files named `SEALED_*` are the dev_human tool
predictions: do not open them before the tool re-adjudication.

In [ ]:
# ---- parameters ----
STAGE = "sweep"            # "sweep" or "finals"
KAGGLE_DATASET = "group4-laya-parity"
MIN_FREE_GB = 12

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, time

found = sorted(pathlib.Path("/kaggle/input").rglob(f"laya_parity_plan_{STAGE}.json"))
assert found, f"attach the dataset {KAGGLE_DATASET} with laya_parity_plan_{STAGE}.json"
DATA = found[0].parent
PLAN_PATH = found[0]
plan = json.loads(PLAN_PATH.read_text())
OUT = pathlib.Path("/kaggle/working/laya_parity")
PRED = pathlib.Path("/kaggle/working/predictions")
LOGS = pathlib.Path("/kaggle/working/logs")
for d in (OUT, PRED, LOGS):
    d.mkdir(parents=True, exist_ok=True)
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
print("GPUs:", gpus)
assert len(gpus) >= len(plan["queues"]), f"{len(plan['queues'])} queues need {len(plan['queues'])} GPUs"
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"free output space: {free_gb:.0f} GB; plan made from commit {plan['meta']['repo_commit'][:10]}",
      "(working tree dirty)" if plan["meta"]["repo_dirty"] else "")
assert free_gb >= MIN_FREE_GB

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

for name, expected in plan["files"].items():
    assert sha256(DATA / name) == expected, f"{name}: hash mismatch"
print(f"{len(plan['files'])} files ok; queues: {plan['queues']}")

In [ ]:
# Laya at the pinned version and its base checkpoint at the pinned revision.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"laya=={plan['laya_version']}"], check=True)
import torch, transformers, laya  # noqa: E402
from huggingface_hub import snapshot_download  # noqa: E402
assert torch.cuda.is_available() and torch.cuda.device_count() >= len(plan["queues"])
BASE = pathlib.Path(snapshot_download(plan["base"]["repo"], revision=plan["base"]["revision"]))
print({"torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
       "laya": laya.__version__, "base": str(BASE)})
assert laya.__version__ == plan["laya_version"]

In [ ]:
# Both queues at once, one per GPU; logs on persistent storage.
procs = []
for gpu, queue in enumerate(plan["queues"]):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    log = open(LOGS / f"{STAGE}_gpu{gpu}.log", "a")
    cmd = [sys.executable, str(DATA / "laya_parity_runner.py"), str(PLAN_PATH), str(DATA), str(BASE),
           str(OUT), str(PRED), *queue]
    procs.append((gpu, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    print(f"GPU {gpu}: {queue}")
start = time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(120)
    done = sorted(d.name for d in OUT.iterdir() if (d / "DONE").exists())
    print(f"{(time.time() - start) / 60:5.1f} min; done: {done}", flush=True)
for gpu, p, log in procs:
    log.close()
    print(f"GPU {gpu} exit code {p.returncode}")

In [ ]:
# Summary; predictions zipped for a small download.
rows = []
for run in plan["runs"]:
    env = OUT / run["run_id"] / "environment.json"
    if env.exists():
        r = json.loads(env.read_text())
        rows.append((run["run_id"], "ok", r["train_seconds"], r["predict_seconds"], r["weights_sha256"][:12]))
    else:
        failed = sorted(OUT.glob(run["run_id"] + ".failed-*"))
        rows.append((run["run_id"], "FAILED" if failed else "missing", None, None, None))
for row in rows:
    print(*row, sep="\t")
shutil.make_archive(f"/kaggle/working/predictions_{STAGE}", "zip", PRED)
for gpu in range(len(plan["queues"])):
    print(f"--- GPU {gpu} log tail ---")
    print((LOGS / f"{STAGE}_gpu{gpu}.log").read_text()[-2000:])
assert all(r[1] == "ok" for r in rows), "some runs did not finish: rerun the notebook (finished runs are skipped)"

## After the session

Download `predictions_<STAGE>.zip` first (scoring needs only these), then the checkpoints the
protocol names (served seeds; Section 5). Unzip predictions into `outputs/predictions/laya_parity/`
on the machine with the repository and score with `scripts/score_laya_parity.py`.